# 3. Writing a custom policy

Any object with `act` / `learn` / `maybe_checkpoint` is a valid agent. Here we write a trivial baseline — always place the six highest-`base` widgets — and run it through the same environment and runner as the built-in policies.

In [ ]:
import os, sys
# notebooks live in notebooks/; make the repo root importable
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
from edp.env import PageCompositionEnv
from edp.agents import run_episode
from edp.policies.edp import make_modules
from edp.config import N_SLOTS

class TopBaseAgent:
    '''Always place the N_SLOTS widgets with the highest base prior.'''
    def __init__(self):
        m = make_modules()
        self.page = sorted(m, key=lambda w: -m[w]['base'])[:N_SLOTS]
    def act(self, obs):
        return self.page, None      # payload=None -> no learning signal
    def learn(self, reward, payload):
        pass
    def maybe_checkpoint(self, index):
        pass

In [ ]:
env = PageCompositionEnv(n=2000, seed=42, source='parametric')
out = run_episode(env, TopBaseAgent())
print('TopBaseAgent regret% =', round(out['regret_pct'], 2))

## A learning custom policy

To learn online, return a `payload` from `act` (anything you need to attribute the delayed reward later) and update in `learn`. The environment hands each matured `(reward, payload)` pair back to `learn` once the delay elapses. Below, an epsilon-greedy-ish placeholder records the running mean reward per first-slot widget.

In [ ]:
import random
from edp.catalog import WIDGETS
from edp.config import N_SLOTS

class GreedyFirstSlot:
    def __init__(self, seed=0):
        self.mean = {w: 0.0 for w in WIDGETS}
        self.n = {w: 0 for w in WIDGETS}
        self.rng = random.Random(seed)
    def act(self, obs):
        if self.rng.random() < 0.1:           # explore
            first = self.rng.choice(WIDGETS)
        else:                                  # exploit best mean
            first = max(self.mean, key=self.mean.get)
        rest = [w for w in WIDGETS if w != first][:N_SLOTS - 1]
        page = [first] + rest
        return page, first                     # payload = the slot-1 arm
    def learn(self, reward, first):
        self.n[first] += 1
        self.mean[first] += (reward - self.mean[first]) / self.n[first]
    def maybe_checkpoint(self, index):
        pass

env = PageCompositionEnv(n=3000, seed=42, source='parametric')
out = run_episode(env, GreedyFirstSlot(seed=1))
print('GreedyFirstSlot regret% =', round(out['regret_pct'], 2))

Neither toy is competitive with the built-in policies — that is the point. The env/agent boundary lets you drop in any idea and measure it against the same oracle, under the same production reward stack, with three lines of glue.